# Aquaculture Model Training

This notebook demonstrates the training workflow for the aquaculture ML framework using actual competition data.


## 1. Setup and Configuration


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.feature_selection import FeatureSelector, select_temporal_features, select_metadata_features
from src.config import TrainingConfig
from src.trainer import Trainer

# For reproducibility
np.random.seed(42)
random.seed(42)

# Define where your data files live.
# Adjust this path if your data are located elsewhere.
DATA_DIR = Path("../data")      # relative to the notebook's working directory
# Verify the directory exists
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

# Define the path to the experiment directory where models and results will be saved
EXPERIMENT_DIR = Path("../experiments")  # relative to the notebook's working directory
# Create the experiment directory if it doesn't exist
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data Loading and Preparation


In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X = train_df[feature_cols].values

# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


## 3. Model Training and Evaluation


In [ ]:
# Create a configuration objec#t
config = TrainingConfig()                     # <-- instantiate the config3

# Select model type (e.g., 'lightgbm', 'catboost', or 'xgboost')
#config.model_type = 'lightgbm'                # <-- choose your model type
config.model_type = 'catboost'
#config.model_type = 'xgboost'

# Select enginerring features to us#e
config.feature_engineering_config.include_optical = True
config.feature_engineering_config.include_sar = True
config.feature_engineering_config.include_temporal_statistics = True
config.feature_engineering_config.include_cross_sensor_features = True
config.feature_engineering_config.include_metadata = False
config.feature_engineering_config.include_normalized_optical = False
config.feature_engineering_config.include_directional_vote = True
config.feature_engineering_config.include_conditional_features = True

# Feature selection configuration (disabled by default, meaning all features are used)
# To enable feature selection, set feature_selection_enabled = True
# To customize feature selection, modify feature_selection_method and feature_sele#ction_kwargs
config.feature_selection_enabled = False      # Set to True to enable feature sele#ction
config.feature_selection_method = 'groups'    # Method to use for feature selectio#n
config.feature_selection_kwargs = {}          # Keyword arguments for the selection method#

# Enable SHAP
config.compute_shap = True
config.shap_sample_size = 1821

# Set experiment directory in config
config.experiment_dir = EXPERIMENT_DIR

# Set conditional features configuration
config.feature_engineering_config.conditional_feature_specs = [
        {
            "base_feature": "NDWI_max",
            "thresholds": [0.0],
            "outputs": [0, 1]  # NDWI_max < 0 -> 0, NDWI_max >= 0 -> 1
        },
        {
            "base_feature": "VH_NDWI_ratio_min",
            "thresholds": [0.0],
            "outputs": [0, 1]  # VH_NDWI_ratio_min < 0 -> 0, VH_NDWI_ratio_min >= 0 -> 1
       },
        {
            "base_feature": "VV_NDWI_ratio_min",
            "thresholds": [0.0],
            "outputs": [0, 1]  # VV_NDWI_ratio_min < 0 -> 0, VV_NDWI_ratio_min >= 0 -> 1
        },
        {
            "base_feature": "MNDWI_max",
            "thresholds": [0.0],
            "outputs": [0, 1]  # MNDWI_max < 0 -> 0, MNDWI_max >= 0 -> 1
       },
        {
            "base_feature": "NDVI_mean",
            "thresholds": [0.1],
           "outputs": [0, 1]  # NDVI_mean < 0.1 -> 0, NDVI_mean >= 0.1 -> 1
        }
    ]


# set feature selection configuration (uncomment to enable and customize)
#config.feature_selection_enabled = True
#config.feature_selection_method = 'groups'
#config.feature_selection_kwargs = {'groups': ['temporal', 'directional_vote', 'conditional']}

config.feature_selection_enabled = True       # turn the selector ON
config.feature_selection_method = 'names'     # select features by their exact names
config.feature_selection_kwargs = {
    'names': [
        # ---- the exact features you want to keep ----
        'directional_vote_fraction_positive',
        'directional_vote_fraction_ge_2',
        'directional_vote_fraction_eq_4',
        'directional_vote_mean',
        'directional_vote_min',
        'directional_vote_max',
        'NDWI_max_cond_0',
        'VH_NDWI_ratio_min_cond_1',
        'VV_NDWI_ratio_min_cond_2',
        'MNDWI_max_cond_3',
        'NDVI_mean_cond_4',
        'VH_NDVI_ratio_slope',
        'conditional_vote',
        'conditional_vote_directional_mean',
        'VH_NDWI_ratio_mean'
    ]
}

#config.feature_selection_enabled = True
#config.feature_selection_method = 'combine'
#config.feature_selection_kwargs = {
#    'include': {'method': 'patterns', 'patterns': ['.*']},  # Match all features
#    'exclude': {'method': 'names', 'names': ['NDWI_max']}   # Exclude NDWI_max
#}

#config.feature_selection_enabled = True
#config.feature_selection_method = 'combine'
#config.feature_selection_kwargs = {
#    'include': {'method': 'groups', 'groups': ['temporal', 'directional_vote']},  # Match all features
#    'exclude': {'method': 'patterns', 'patterns': [
#        'green_.*',
#        'nir_.*',
#        'nira_.*',
#        'swir1_.*',
#        'swir2_.*',
#        'MNDWI_max',
#        'NDMI_std',
#        'NDMI_max',
#        'NDWI_mean',
#        'MNDWI_std',
#        'VV_NDWI_ratio_min',
#        'MNDWI_min',
#        'VH_VV_ratio_mean',
#        'VH_NDVI_mul_max',
#        'NDWI_max',
#        'MNDWI_mean',
#        'NDRE2_min',
#        'VH_NDWI_mul_mean'
#        ]}   # Exclude optical band features (green_*, nir_*, nira_*, swir1_*, swir2_*)
# Feature selection information
print("Feature selection configuration:")
print(f"  Enabled: {config.feature_selection_enabled}")
print(f"  Method: {config.feature_selection_method}")
print(f"  Kwargs: {config.feature_selection_kwargs}")
if config.feature_selection_enabled:
    print("  Feature selection is ENABLED - only selected features will be used for training")
else:
    print("  Feature selection is DISABLED - all features will be used for training (default behavior)")


# Initialize trainer
trainer = Trainer(config)

In [ ]:
# Train models
print("Training models...")
trainer.fit(X, y)

# Save the trainer (including feature selector) for later use in inference
print("Saving trainer for later use...")
trainer.save()
print("Trainer saved successfully!")

In [ ]:
# Evaluate training performance with observation simulation (matches training conditions)
print("\n=== Training set evaluation (with observation stimulation) ===")
train_preds = trainer.predict(X, training=True)
train_probas = trainer.predict_proba(X, training=True)[:, 1]

from src.metrics import calculate_metrics, competition_score
metrics = calculate_metrics(y, train_probas)
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

comp = competition_score(y, train_probas)
print(f"  competition_score: {comp:.4f}")
